# Tea Flush-Stage Classifier — the WHEN dimension

Maps a tea-leaf photo to a crop stage, to populate the **WHEN** axis of the
WHO×WHERE×WHEN metric. Two classes for now (`dormant` is deferred — no dataset yet):

| idx | class | risk |
|-----|-------|------|
| 0 | `new_flush` | **HIGH** — prime Helopeltis window |
| 1 | `mature_leaf` | low |

Architecture: **MobileNetV2** (ImageNet) with a 2-class head. Training is
implemented reproducibly in [`src/train_flush_stage.py`](../src/train_flush_stage.py);
this notebook **loads the trained model and shows its results inline** (nothing is saved).

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

import random
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

from src.train_flush_stage import (
    CLASSES, DATA_DIR, MODELS_DIR, SEED, build_model, eval_tf, gather_samples,
)
print("Classes:", CLASSES)

## 2. Data

Dataset: TeaLeafAgeQuality (Mendeley, CC BY 4.0). Not versioned; see [`data/raw/flush_stages/README.md`](../data/raw/flush_stages/README.md).

In [ ]:
counts = {c: len(list((DATA_DIR / c).glob("*.jpg"))) for c in CLASSES}
print("Images per class:", counts)
print("Total:", sum(counts.values()))

## 3. Load the trained model

Train (a few minutes on CPU) with:
```
python -m src.train_flush_stage --epochs-head 5 --epochs-ft 2
```
Then load the weights here.

In [ ]:
model = build_model(num_classes=len(CLASSES), freeze_backbone=False)
model.load_state_dict(torch.load(MODELS_DIR / "flush_stage.pt", map_location="cpu"))
model.eval()
print("Loaded", MODELS_DIR / "flush_stage.pt")


@torch.no_grad()
def predict(path):
    """Return (class_index, confidence) for one image path."""
    probs = torch.softmax(model(eval_tf(Image.open(path).convert("RGB")).unsqueeze(0)), 1)[0]
    return int(probs.argmax()), float(probs.max())

## 4. Results on the held-out test set

Same deterministic split as training (fixed seed), so these images were never seen during training.

In [ ]:
samples = gather_samples()
labels = [y for _, y in samples]
_, temp = train_test_split(samples, test_size=0.30, stratify=labels, random_state=SEED)
_, test = train_test_split(temp, test_size=0.50,
                           stratify=[y for _, y in temp], random_state=SEED)

y_true = [y for _, y in test]
y_pred = [predict(p)[0] for p, _ in test]
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))

In [ ]:
# Confusion matrix (inline)
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(CLASSES)), CLASSES)
ax.set_yticks(range(len(CLASSES)), CLASSES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Confusion matrix (test, n={len(test)})")
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Prediction grid (inline) - green = correct, red = wrong
random.seed(0)
sample = random.sample(test, 12)
fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for ax, (path, true) in zip(axes.ravel(), sample):
    pred, conf = predict(path)
    ok = pred == true
    ax.imshow(Image.open(path).convert("RGB").resize((224, 224)))
    title = "pred: {} ({:.2f})".format(CLASSES[pred], conf) + chr(10) + "true: " + CLASSES[true]
    ax.set_title(title, color="#1b7a1b" if ok else "#c0271e", fontsize=10)
    for s in ax.spines.values():
        s.set_edgecolor("#1b7a1b" if ok else "#c0271e"); s.set_linewidth(3)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## Next steps

1. Improve `new_flush` recall (the high-risk class): class-weighted loss or a dedicated decision threshold.
2. Add the `dormant` class (dataset from iNaturalist) → move to 3 classes.
3. Validate on real field photos from the plantation.